In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        _tok = ""
        try:
            from google.colab import userdata
            _tok = userdata.get("GH_TOKEN") or ""
        except Exception:
            _tok = ""
        if not _tok:
            print("WARNING: no 'GH_TOKEN' Colab secret found; cloning this PRIVATE repo will fail.\n"
                  "Add a GitHub token (repo scope) via the key icon (Secrets) as 'GH_TOKEN', then re-run.")
        _url = (f"https://{_tok}@github.com/{_slug}.git" if _tok
                else f"https://github.com/{_slug}.git")
        subprocess.run(["git", "clone", "--depth", "1", _url, str(_root)], check=True)
        subprocess.run(["git", "-C", str(_root), "remote", "set-url", "origin",
                        f"https://github.com/{_slug}.git"])  # keep the token out of the saved remote
    os.chdir(_root / "07-application-agent-framework/long-running-durable/long-running-agentic/long-running-agents-gcp/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 02 · Fan-out / fan-in — practice
Reference: `notebooks/solutions/ex2_fan_out_fan_in.py`.

In [ ]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))          # repo root when run from notebooks/
sys.path[:0] = [os.path.join(ROOT, "src"), os.path.join(ROOT, "notebooks")]

def show_journal(run):
    print(f"run {run.run_id}  status={run.status.value}  version={run.version}  steps={run.usage.steps}  tokens={run.usage.tokens}  cost=${run.usage.cost_usd:.4f}")
    for s in run.journal:
        out = json.dumps(s.output, default=str)[:70] if s.output is not None else (s.error or "")
        print(f"  [{s.index}] {s.kind.value:<6} {s.status.value:<7} {s.name:<18} key={s.idempotency_key or '-':<20} {out}")

In [ ]:
from lragents.core import *
from lragents.core.transport import Envelope
from lragents.patterns import FanOutFanIn
from lragents.practice_checks import check_fan_in_mutate, check_fan_out_fan_in

## Exercise 1 — the fan-in transaction function
`make_mutate(subtask_id, result)` returns `mutate(run) -> bool`. It must:
* record `result` under `run.state["fan"]["results"][subtask_id]` and increment `completed` — **only if not already recorded** (duplicate delivery);
* return `True` iff `completed == expected` — including for a *late duplicate* that arrives after completion (why?).

In [ ]:
def make_mutate(subtask_id, result):
    def mutate(run):
        fan = run.state["fan"]
        # TODO
        raise NotImplementedError
    return mutate

In [ ]:
print(check_fan_in_mutate(make_mutate))

## Exercise 2 — the worker handler
Subclass `FanOutFanIn` and implement `handle_subtask`. Order matters:
1. run the worker **idempotently** (`run_idempotent(self.idem, key, ...)`) — outside any transaction;
2. `run, all_done = transact(self.store, run_id, make_mutate(subtask_id, result))`;
3. if `all_done`: `self.dispatcher.enqueue(Envelope(run_id, 1, "aggregate"))`.

In [ ]:
class MyFan(FanOutFanIn):
    def handle_subtask(self, run_id, subtask_id, task):
        key = f"{run_id}:sub:{subtask_id}"
        # TODO
        raise NotImplementedError

In [ ]:
print(check_fan_out_fan_in(MyFan))

## Exercise 3 — design questions
1. You have 500 subtasks. What breaks first in this design on Firestore, and what are two fixes?
2. A worker takes 20 minutes. Which GCP service should run it and why (Cloud Run service vs job vs Workflows)?
3. The aggregator LLM call fails with a 429. What retries it, and what guarantees only one synthesis is written?

In [ ]:
answers = '''
1.
2.
3.
'''

In [ ]:
import inspect
from solutions import ex2_fan_out_fan_in as ref
print(inspect.getsource(ref.make_mutate)); print(inspect.getsource(ref.MyFan))